In [130]:
# Working with data
import pandas as pd
import numpy as np

# src files
from data.load import load_data
from data.config import preprocess

# split train and test
from sklearn.model_selection import train_test_split

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# encode
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# models XGBoost
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# metrics
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import f1_score, accuracy_score, recall_score

# time
import time

In [131]:
df = load_data("Churn_Modelling.xls")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [132]:
# drop columns
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [133]:
df["log_balance"] = np.log1p(df["Balance"])
df.drop("Balance", axis=1, inplace=True)

In [134]:
df["balance_per_product"] = df["log_balance"] / (df["NumOfProducts"] + 1)

In [135]:
df["tenure_age_ratio"] = df["Tenure"] / (df["Age"] + 1)

In [136]:
df["is_high_salary"] = (df["EstimatedSalary"] > df["EstimatedSalary"].median()).astype(int)

In [137]:
experiment = "IsHighSalary",
features =  "is_high_..",

In [138]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,log_balance,balance_per_product,tenure_age_ratio,is_high_salary
0,619,France,Female,42,2,1,1,1,101348.88,1,0.000000,0.000000,0.046512,1
1,608,Spain,Female,41,1,1,0,1,112542.58,0,11.336294,5.668147,0.023810,1
2,502,France,Female,42,8,3,1,0,113931.57,1,11.980813,2.995203,0.186047,1
3,699,France,Female,39,1,2,0,0,93826.63,0,0.000000,0.000000,0.025000,0
4,850,Spain,Female,43,2,1,1,1,79084.10,0,11.740155,5.870078,0.045455,0


In [139]:
# Spliting target column
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Train/test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=23, stratify=y)

# Spliting num and cat cols
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(include="object").columns

In [140]:
# pipelin for numeric cols
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# pipeline for cat cols
cat_pipeline = Pipeline([
    ("encode", OneHotEncoder())
])

# Transform
col_trans = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)

In [141]:
# List of models
models = {
    "Random Forest": RandomForestClassifier(),
    "Logistic Regression": LogisticRegression()
}

In [142]:
# train models
results = []

for name, model in models.items():

    train_model = Pipeline([
        ("transfrom", col_trans),
        ("model", model)
    ])

    #train model
    start_tr = time.time()
    train_model.fit(X_train, y_train)
    end_tr = time.time()

    # predict
    start_pr = time.time()
    y_pre = train_model.predict(X_test)
    end_pr = time.time()

    accuracy = accuracy_score(y_test, y_pre)
    f1 = f1_score(y_test, y_pre)
    recll = recall_score(y_test, y_pre)
    train_time = start_tr - end_tr
    predict_time = start_pr - end_pr 

    # test
    results.append({
        "experiment": experiment,
        "features": features,
        "name": name,
        "f1-score": f1,
        "accuracy": accuracy,
        "recall": recll,
        "train_time": train_time,
        "predict_time": predict_time
    })

# result to df
result_df = pd.DataFrame(results)

c:\Users\ibnmu\anaconda3\envs\churn_env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [143]:
result_df

,experiment,features,name,f1-score,accuracy,recall,train_time,predict_time
0,"(IsHighSalary,)","(is_high_..,)",Random Forest,0.576410,0.862333,0.459902,-0.872336,-0.031655
1,"(IsHighSalary,)","(is_high_..,)",Logistic Regression,0.140496,0.792000,0.083470,-0.033729,-0.007682


In [144]:
import os

file_path = "../outputs/experiment_log.csv"

if os.path.exists(file_path):
    old = pd.read_csv(file_path)
    new = pd.concat([old, result_df], ignore_index=True)
    new.to_csv(file_path, index=False)
else:
    result_df.to_csv(file_path, index=False)

In [145]:
df_outputs = pd.read_csv("../outputs/experiment_log.csv")
df_outputs

,experiment,features,name,f1-score,accuracy,recall,train_time,predict_time
0,"('baseline',)","('raw',)",Random Forest,0.577236,0.861333,0.464812,-0.597237,-0.035895
1,"('baseline',)","('raw',)",Logistic Regression,0.127424,0.790000,0.075286,-0.036628,-0.006511
2,"('HasBalance added',)","('Hasbalnce',)",Random Forest,0.581781,0.865333,0.459902,-0.624565,-0.036839
3,"('HasBalance added',)","('Hasbalnce',)",Logistic Regression,0.127424,0.790000,0.075286,-0.038787,-0.004182
4,"('Balance loged',)","('log_balance',)",Random Forest,0.578462,0.863000,0.461538,-0.642094,-0.034741
5,"('Balance loged',)","('log_balance',)",Logistic Regression,0.286756,0.804333,0.193126,-0.049370,-0.008136
6,"('Balance per Product',)","('balance_per_product',)",Random Forest,0.572606,0.861667,0.454992,-0.731717,-0.044333
7,"('Balance per Product',)","('balance_per_product',)",Logistic Regression,0.164199,0.793000,0.099836,-0.049736,0.000000
8,"('log_balance / numofproducts',)","('balance_per_product',)",Random Forest,0.584270,0.864333,0.468085,-0.703691,-0.036221
9,"('log_balance / numofproducts',)","('balance_per_product',)",Logistic Regression,0.126404,0.792667,0.073650,-0.044562,-0.002127
